
# KOA Onboarding from UFO

Este notebook lê arquivos de modelos de contratos .pl e faz o onboarding desses contratos com base em uma ontologia de fundamentação, neste caso UFO_Service_Contract_Ontology.pl.

Entradas: modelos .pl no diretório configurado.

Saídas: arquivos KOA_UFO_<contrato>.pl no diretório de persistência.


In [1]:
# Monta o Google Drive no ambiente Colab para acessar os arquivos de contratos e salvar os .pl gerados
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!pip uninstall google-generativeai -y
!pip install google-genai

Found existing installation: google-generativeai 0.8.6
Uninstalling google-generativeai-0.8.6:
  Successfully uninstalled google-generativeai-0.8.6


In [3]:
# Importa bibliotecas padrão e do Gemini
import os
from pathlib import Path
import re
import json
import time

from collections import Counter, defaultdict
from typing import Dict, Any, List, Optional, Tuple

from google import genai

In [4]:
# Recupera chave Gemini com secrets do Colab
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [5]:
# Pasta com os modelos Prolog (.pl) do contrato
directory = '/content/drive/My Drive/KOA/onboarding/contratos_SEM_UFO'

# Pasta de saída (onde serão gerados os novos .pl UFO-only)
persist_directory = '/content/drive/My Drive/KOA/onboarding/contratos_COM_UFO'

# Arquivo da ontologia base UFO (Prolog)
BASE_UFO_ONTOLOGY_FILE = '/content/drive/MyDrive/KOA/onboarding/modelos/UFO_Service_Contract_Ontology_v2.pl'

In [6]:
# Arquivos a serem processados
PROLOG_INPUT_FILES = sorted(Path(directory).glob("*.pl"))

print(f"{len(PROLOG_INPUT_FILES)} arquivos encontrados:")
for f in PROLOG_INPUT_FILES:
    print(" -", f.name)

print("BASE_ONTOLOGY_FILE:", BASE_UFO_ONTOLOGY_FILE)

66 arquivos encontrados:
 - KOA_Contrato_OCS_008_2022_-_Hitachi_Vantara_Storage.pl
 - KOA_Contrato_OCS_011_2022_-_Zoom_(Storage_Huawei).pl
 - KOA_Contrato_OCS_011_2023_-_Multiplus_(BIG-IP).pl
 - KOA_Contrato_OCS_012_2022_-_VS_Data_Storages_de_backup.pl
 - KOA_Contrato_OCS_018_2023_-_Asper.pl
 - KOA_Contrato_OCS_023_2024_-_Soluti_(e-CPF).pl
 - KOA_Contrato_OCS_026_2024_-_Certificados_INFOCONV,_eSocial_e_e-CNPJ_(Soluti).pl
 - KOA_Contrato_OCS_027_2024_-_OpF_CMP.pl
 - KOA_Contrato_OCS_027_2024_-_SOLUTI_Certificados_digitais_SPB_e_OpF.pl
 - KOA_Contrato_OCS_028_2022_-_Telsinc.pl
 - KOA_Contrato_OCS_028_2024_-_G4F_SOLUCOES_CORPORATIVAS_LTDA.pl
 - KOA_Contrato_OCS_032_2024_-_VSDATA_Suporte_MQ.pl
 - KOA_Contrato_OCS_035_2022_-_Claro.pl
 - KOA_Contrato_OCS_042_2021_-_TIVIT_(Data_Center_Alternativo).pl
 - KOA_Contrato_OCS_043_2021_-_Rational.pl
 - KOA_Contrato_OCS_044_2022_-_SAP_IDM_Conectores.pl
 - KOA_Contrato_OCS_046_2021_-_Ingram_(Suporte_Notes).pl
 - KOA_Contrato_OCS_046_2024_-_SENSEDIA_S.

In [7]:
# Funções para tratamento de chamadas ao Gemini
def call_gemini_json(model, prompt: str, max_retries: int = 3, sleep_s: float = 2.0):
    """Chama Gemini e retorna um dict JSON (ou levanta erro)."""
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'))
            resp = client.models.generate_content(
                model=model,
                contents=prompt
            )
            text = getattr(resp, "text", None) or (resp.candidates[0].content.parts[0].text if getattr(resp, "candidates", None) else None)
            if not text:
                raise ValueError("Gemini retornou resposta vazia.")
            # tenta localizar JSON na resposta
            m = re.search(r"\{.*\}", text, flags=re.DOTALL)
            json_text = m.group(0) if m else text
            return json.loads(json_text)
        except Exception as e:
            last_err = e
            if attempt < max_retries:
                time.sleep(sleep_s)
            else:
                raise

In [8]:
# Funções genéricas de apoio
def derive_contract_id_from_path(p: Path) -> str:
    """
    Extrai um ID de contrato estável a partir do nome do arquivo.
    Ex.: KOA_048_2022_ETL.pl  -> contract_ocs_048_2022
    """
    stem = p.stem.lower()

    m = re.search(r"(\d{3})_(\d{4})", stem)
    if m:
        return f"contract_ocs_{m.group(1)}_{m.group(2)}"

    # fallback: usa o próprio stem (sanitizado)
    stem = re.sub(r"[^a-z0-9_]+", "_", stem).strip("_")
    return f"contract_{stem}"

## Etapa 1 — Carregar ontologia de fundamentação (UFO-only)

Nesta versão vamos trabalhar **somente** com:
- a ontologia de fundamentação (UFO Service Contract)
- os modelos Prolog dos contratos (metadados, cláusulas e facts extraídos)

A ontologia é usada para **restringir** o que o LLM pode sugerir (evitar alucinações) e para **validar** o resultado.

In [10]:
with open(BASE_UFO_ONTOLOGY_FILE, "r", encoding="utf-8") as f:
    ufo_text = f.read()

UFO_CLASSES = sorted(set(re.findall(r'^class\(([^)]+)\)\.', ufo_text, flags=re.M)))

# Subconjunto típico de "legal moments" (ajuste se a sua ontologia tiver outros nomes)
DEFAULT_LEGAL_MOMENTS = {
    "duty_to_act","duty_to_omit",
    "right_to_action","right_to_omission",
    "permission_to_act","permission_to_omit",
    "power","subjection","immunity",
    "no_right_to_action","no_right_to_omission","disability"
}
UFO_LEGAL_MOMENT_CLASSES = sorted([c for c in UFO_CLASSES if c in DEFAULT_LEGAL_MOMENTS])

print("Ontologia:", Path(BASE_UFO_ONTOLOGY_FILE).name)
print("Total de classes:", len(UFO_CLASSES))
print("Legal moments:", UFO_LEGAL_MOMENT_CLASSES)

Ontologia: UFO_Service_Contract_Ontology_v2.pl
Total de classes: 43
Legal moments: ['disability', 'duty_to_act', 'duty_to_omit', 'immunity', 'no_right_to_action', 'no_right_to_omission', 'permission_to_act', 'permission_to_omit', 'power', 'right_to_action', 'right_to_omission', 'subjection']


## Etapa 2 — Parser pragmático do Prolog do contrato

O seu Prolog de contrato tem padrões como:

- `contract/1`
- `contract_metadata/3`
- `contract_clause/4`
- `contract_clause_fact/5`

Aqui fazemos um parser simples (regex) suficiente para:
- identificar o `contract_id`
- extrair metadados (contratante/contratado/partes etc.)
- listar cláusulas e fatos por cláusula

In [11]:
CONTRACT_RE = re.compile(r'^\s*contract\(([^)]+)\)\.\s*$', re.M)
META_RE = re.compile(r'^\s*contract_metadata\(([^,]+),\s*([^,]+),\s*(.+)\)\.\s*$', re.M)
CLAUSE_RE = re.compile(r'^\s*contract_clause\(([^,]+),\s*([^,]+),\s*(.+?),\s*(.+)\)\.\s*$', re.M)
CLAUSE_FACT_RE = re.compile(r'^\s*contract_clause_fact\(([^,]+),\s*([^,]+),\s*([^,]+),\s*(.+?),\s*(.+)\)\.\s*$', re.M)

def parse_contract_prolog(text: str) -> Dict[str, Any]:
    m = CONTRACT_RE.search(text)
    if not m:
        raise ValueError("Não encontrei fact contract(CONTRACT_ID).")
    contract_id = m.group(1).strip()

    metadata: Dict[str, List[str]] = {}
    for m in META_RE.finditer(text):
        cid, key, val = m.group(1).strip(), m.group(2).strip(), m.group(3).strip()
        if cid != contract_id:
            continue
        metadata.setdefault(key, []).append(val)

    clauses: Dict[str, Dict[str, Any]] = {}
    for m in CLAUSE_RE.finditer(text):
        cid, clause_id = m.group(1).strip(), m.group(2).strip()
        if cid != contract_id:
            continue
        clauses[clause_id] = {"title": m.group(3).strip(), "body": m.group(4).strip(), "facts": []}

    for m in CLAUSE_FACT_RE.finditer(text):
        cid, clause_id, fact_key = m.group(1).strip(), m.group(2).strip(), m.group(3).strip()
        if cid != contract_id:
            continue
        clauses.setdefault(clause_id, {"title": None, "body": None, "facts": []})
        clauses[clause_id]["facts"].append({
            "key": fact_key,
            "value": m.group(4).strip(),
            "evidence": m.group(5).strip()
        })

    return {"contract_id": contract_id, "metadata": metadata, "clauses": clauses}

def read_contract_file(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return parse_contract_prolog(f.read())

## Etapa 3 — Micro-tarefa 1: normalizar partes e atribuir papéis (roles)

O LLM recebe metadados e retorna JSON com:
- agentes normalizados (`agent_id`, `label`, `cnpj`)
- papéis (`service_customer_role`, `hired_service_provider_role`)

**Prompts em inglês**.

In [12]:
ROLE_GROUNDING_PROMPT = r"""
You will receive a Brazilian public service contract metadata snippet.
Your task: normalize party identifiers and assign UFO roles.

Output ONLY valid JSON with this schema:
{{
  "contract_id": "<same as input>",
  "agents": [
    {{"agent_id": "lower_snake_case_id", "label": "original name string", "cnpj": "optional string or null"}}
  ],
  "roles": [
    {{"agent_id": "agent_id", "role_class": "service_customer_role|hired_service_provider_role", "contract_id": "<contract_id>"}}
  ]
}}

Constraints:
- agent_id must be lowercase snake_case, ASCII only.
- Use role_class exactly as given.
- Prefer using metadata keys 'contratante' and 'contratado'. If missing, infer from 'partes' order only as a fallback.
- Do not invent parties not present in the input.

INPUT:
contract_id: {contract_id}
metadata_json: {metadata_json}
""".strip()

def build_role_grounding(contract: Dict[str, Any], gemini_model) -> Dict[str, Any]:
    prompt = ROLE_GROUNDING_PROMPT.format(
        contract_id=contract["contract_id"],
        metadata_json=json.dumps(contract["metadata"], ensure_ascii=False),
    )
    return call_gemini_json(gemini_model, prompt)


## Etapa 4 — Micro-tarefa 2: candidatos de momentos jurídicos por cláusula

Para cada cláusula, o LLM devolve candidatos:
- `legal_moment_class` (restrito ao conjunto permitido)
- `bearer_hint` (quem porta: contratante/contratado/agent_id/unknown)
- `action_id` e `action_label` (em inglês)
- `evidence` (até 25 palavras)
- `confidence`

**Prompts em inglês**.

In [13]:
LEGAL_MOMENTS_PROMPT = r"""
You are grounding a contract clause into UFO legal moments.

Allowed legal moment classes (use exactly one of these):
{allowed_classes}

Output ONLY valid JSON with this schema:
{{
  "clause_id": "{clause_id}",
  "candidates": [
    {{
      "legal_moment_class": "one_of_allowed_classes",
      "bearer_hint": "contracting_party|contracted_party|bndes|supplier|<agent_id if known>|unknown",
      "action_id": "lower_snake_case_action_id",
      "action_label": "short English label",
      "evidence": "short quote or paraphrase from clause/facts (max 25 words)",
      "confidence": 0.0
    }}
  ]
}}

Rules:
- action_id must be snake_case, ASCII.
- Keep action_label short, in English.
- Prefer creating paired moments when natural (e.g., duty_to_act vs right_to_action), but still output as separate candidates.
- Do NOT add any classes outside the allowed list.
- Do NOT hallucinate facts beyond the clause text/facts.

INPUT:
clause_title: {clause_title}
clause_text: {clause_text}
clause_facts_json: {clause_facts_json}
known_parties_json: {known_parties_json}
""".strip()

def extract_legal_moment_candidates(
    clause_id: str,
    clause: Dict[str, Any],
    known_parties: Dict[str, Any],
    gemini_model,
) -> Dict[str, Any]:
    prompt = LEGAL_MOMENTS_PROMPT.format(
        allowed_classes=", ".join(UFO_LEGAL_MOMENT_CLASSES),
        clause_id=clause_id,
        clause_title=clause.get("title") or "",
        clause_text=clause.get("body") or "",
        clause_facts_json=json.dumps(clause.get("facts", []), ensure_ascii=False),
        known_parties_json=json.dumps(known_parties, ensure_ascii=False),
    )
    return call_gemini_json(gemini_model, prompt)


## Etapa 5 — Catálogo de ações e emissão Prolog (sem LLM)

Sem ontologia de domínio, a consistência é mantida por um catálogo simples de ações.

In [14]:
from dataclasses import dataclass

def normalize_action_id(action_id: str) -> str:
    action_id = (action_id or "").strip().lower()
    action_id = re.sub(r"[^a-z0-9_]+", "_", action_id)
    action_id = re.sub(r"_+", "_", action_id).strip("_")
    return action_id or "unnamed_action"

@dataclass
class ActionCatalogItem:
    action_id: str
    label: str

class ActionCatalog:
    def __init__(self):
        self.items: Dict[str, ActionCatalogItem] = {}

    def ensure(self, action_id: str, label: str) -> str:
        aid = normalize_action_id(action_id)
        if aid not in self.items:
            self.items[aid] = ActionCatalogItem(aid, (label or "").strip()[:120])
        return aid

    def to_prolog(self) -> str:
        lines = []
        for aid in sorted(self.items.keys()):
            lbl = (self.items[aid].label or "").replace("'", "\\'")
            lines.append(f"action_type({aid}).")
            lines.append(f"action_label({aid}, '{lbl}').")
        return "\n".join(lines)

def emit_instance_of(individual: str, cls: str) -> str:
    return f"instance_of({individual}, {cls})."

def emit_plays_role(agent_id: str, role_class: str, contract_id: str) -> str:
    return f"plays_role({agent_id}, {role_class}, {contract_id})."

def emit_clause_of(clause_id: str, contract_id: str) -> str:
    return f"clause_of({clause_id}, {contract_id})."

def emit_legal_relation_instance(clause_id: str, lm_class: str, bearer: str, action_id: str) -> str:
    return f"legal_relation_instance({clause_id}, {lm_class}, {bearer}, {action_id})."

## Etapa 6 — Pipeline por contrato (validação + seleção + geração de arquivos)

Saída:
- `{contract_id}_UFO_only.pl`
- `{contract_id}_trace.json` (rastreabilidade)

> Observação: esta célula **não altera** suas configurações iniciais de caminhos; ela cria uma subpasta de saída automaticamente.

In [17]:
def validate_candidates(candidates: List[Dict[str, Any]], known_agent_ids: List[str]) -> List[str]:
    errors = []
    allowed_bearers = set(known_agent_ids) | {"unknown", "contracting_party", "contracted_party", "bndes", "supplier"}
    allowed_lm = set(UFO_LEGAL_MOMENT_CLASSES)

    for i, c in enumerate(candidates):
        lm = c.get("legal_moment_class")
        if lm not in allowed_lm:
            errors.append(f"cand[{i}]: invalid legal_moment_class: {lm}")

        bearer = c.get("bearer_hint", "unknown")
        if bearer not in allowed_bearers:
            errors.append(f"cand[{i}]: unexpected bearer_hint: {bearer}")

        if not normalize_action_id(c.get("action_id", "")):
            errors.append(f"cand[{i}]: empty action_id")
    return errors

def select_top_candidates(candidates: List[Dict[str, Any]], top_k: int = 12) -> List[Dict[str, Any]]:
    def conf(c):
        try:
            return float(c.get("confidence", 0.0))
        except Exception:
            return 0.0
    return sorted(candidates, key=conf, reverse=True)[:top_k]

def bearer_hint_to_agent_id(bearer_hint: str, role_grounding: Dict[str, Any]) -> str:
    roles = role_grounding.get("roles", [])
    customer = None
    provider = None
    for r in roles:
        if r.get("role_class") == "service_customer_role":
            customer = r.get("agent_id")
        if r.get("role_class") == "hired_service_provider_role":
            provider = r.get("agent_id")

    if bearer_hint in {"contracting_party", "bndes"} and customer:
        return customer
    if bearer_hint in {"contracted_party", "supplier"} and provider:
        return provider

    known = {a.get("agent_id") for a in role_grounding.get("agents", [])}
    if bearer_hint in known:
        return bearer_hint

    return "unknown"

def run_contract_onboarding(contract_path: Path, gemini_model, top_k_per_clause: int = 10) -> Dict[str, Any]:
    contract = read_contract_file(contract_path)
    cid = contract["contract_id"]
    print(f"\n=== Processing {contract_path.name} (contract_id={cid}) ===")

    # 1) roles
    role_grounding = build_role_grounding(contract, gemini_model)
    agents = role_grounding.get("agents", [])
    roles = role_grounding.get("roles", [])
    known_agent_ids = [a["agent_id"] for a in agents if "agent_id" in a]

    # 2) cláusulas
    clause_items = list(contract["clauses"].items())
    catalog = ActionCatalog()
    selected_moments: List[Dict[str, Any]] = []
    clause_outputs: Dict[str, Any] = {}

    for clause_id, clause in clause_items:
        cand_json = extract_legal_moment_candidates(clause_id, clause, role_grounding, gemini_model)
        candidates = cand_json.get("candidates", [])
        errors = validate_candidates(candidates, known_agent_ids)

        if errors:
            print(f"[WARN] validation errors in clause {clause_id}:")
            for e in errors[:10]:
                print("  -", e)

        for c in candidates:
            c["action_id"] = catalog.ensure(c.get("action_id", ""), c.get("action_label", ""))
            c["bearer"] = bearer_hint_to_agent_id(c.get("bearer_hint", "unknown"), role_grounding)

        chosen = select_top_candidates(candidates, top_k=top_k_per_clause)
        for c in chosen:
            if c.get("legal_moment_class") in set(UFO_LEGAL_MOMENT_CLASSES):
                selected_moments.append({
                    "clause_id": clause_id,
                    "legal_moment_class": c.get("legal_moment_class"),
                    "bearer": c.get("bearer", "unknown"),
                    "action_id": c.get("action_id"),
                    "evidence": c.get("evidence", ""),
                    "confidence": c.get("confidence", 0.0),
                })

        clause_outputs[clause_id] = {"raw": cand_json, "selected": chosen, "validation_errors": errors}

    # 3) saída: cria subpasta ao lado da pasta de entrada (sem mexer no que você já configurou)
    input_dir = Path(globals().get("directory", contract_path.parent.as_posix()))
    out_dir = Path(globals().get("persist_directory"))
    out_pl_path = out_dir / f"KOA_UFO_{cid}.pl"
    out_trace_path = out_dir / f"KOA_UFO_{cid}_trace.json"

    # 4) emissão Prolog
    out_lines = []
    out_lines.append(f"% UFO-only grounding generated from: {contract_path.name}")
    out_lines.append(f"% contract_id: {cid}")
    out_lines.append("")
    out_lines.append(emit_instance_of(cid, "legal_service_agreement"))
    out_lines.append("")

    for a in agents:
        out_lines.append(emit_instance_of(a["agent_id"], "agent"))

    out_lines.append("")
    for r in roles:
        out_lines.append(emit_plays_role(r["agent_id"], r["role_class"], r["contract_id"]))

    out_lines.append("")
    for clause_id, _ in clause_items:
        out_lines.append(emit_clause_of(clause_id, cid))

    out_lines.append("")
    for m in selected_moments:
        out_lines.append(emit_legal_relation_instance(m["clause_id"], m["legal_moment_class"], m["bearer"], m["action_id"]))

    out_lines.append("\n% --- Action catalog (local to this contract grounding) ---")
    out_lines.append(catalog.to_prolog())


    # ---------- Combined output file (ontology + generated ABox + original contract ABox) ----------

    # 1) UFO ontology
    with open(BASE_UFO_ONTOLOGY_FILE, "r", encoding="utf-8") as f:
        ufo_ontology_text = f.read().rstrip()

    # 2) generated ABox
    generated_abox_text = ("\n".join(out_lines)).rstrip()

    # 3) original contract ABox (metadata + literal clauses only)
    def _extract_original_contract_abox_text(contract_file: Path) -> str:
        keep_prefixes = ("contract(", "contract_metadata(", "contract_metadata_raw(", "contract_clause(")
        kept = []
        with open(contract_file, "r", encoding="utf-8") as f:
            for line in f:
                s = line.lstrip()
                if s.startswith("%") or not s.strip():
                    continue
                if s.startswith(keep_prefixes):
                    kept.append(line.rstrip())
        return "\n".join(kept).rstrip()

    original_contract_abox_text = _extract_original_contract_abox_text(contract_path)

    final_text = "\n".join([
        f"% ===== KOA Combined Output | contract_id: {cid} =====",
        "",
        "% ===== 1) UFO Ontology =====",
        ufo_ontology_text,
        "",
        "% ===== 2) Generated UFO ABox =====",
        generated_abox_text,
        "",
        "% ===== 3) Original Contract ABox =====",
        original_contract_abox_text,
        "",
        "% ===== END =====",
        ""
    ])

    with open(out_pl_path, "w", encoding="utf-8") as f:
        f.write(final_text)


    with open(out_trace_path, "w", encoding="utf-8") as f:
        json.dump({
            "contract_id": cid,
            "source_file": contract_path.name,
            "role_grounding": role_grounding,
            "selected_moments": selected_moments,
            "clauses": clause_outputs,
        }, f, ensure_ascii=False, indent=2)

    print("Saved:", out_pl_path)
    print("Saved:", out_trace_path)

    return {"contract_id": cid, "out_pl": str(out_pl_path), "trace": str(out_trace_path), "n_moments": len(selected_moments)}

## Etapa 7 — Execução em lote

Esta célula usa as variáveis das células iniciais:
- `PROLOG_INPUT_FILES`
- `gemini_model`

In [18]:
gemini_model='gemini-2.0-flash'

results = []
for p in PROLOG_INPUT_FILES:
    results.append(run_contract_onboarding(p, gemini_model, top_k_per_clause=10))

print("\nResumo:")
for r in results:
    print(f" - {r['contract_id']}: {r['n_moments']} momentos | {r['out_pl']}")


=== Processing KOA_Contrato_OCS_008_2022_-_Hitachi_Vantara_Storage.pl (contract_id=contrato_ocs_008_2022) ===
Saved: /content/drive/My Drive/KOA/onboarding/contratos_COM_UFO/KOA_UFO_contrato_ocs_008_2022.pl
Saved: /content/drive/My Drive/KOA/onboarding/contratos_COM_UFO/KOA_UFO_contrato_ocs_008_2022_trace.json

=== Processing KOA_Contrato_OCS_011_2022_-_Zoom_(Storage_Huawei).pl (contract_id=contrato_ocs_0011_2022) ===
Saved: /content/drive/My Drive/KOA/onboarding/contratos_COM_UFO/KOA_UFO_contrato_ocs_0011_2022.pl
Saved: /content/drive/My Drive/KOA/onboarding/contratos_COM_UFO/KOA_UFO_contrato_ocs_0011_2022_trace.json

=== Processing KOA_Contrato_OCS_011_2023_-_Multiplus_(BIG-IP).pl (contract_id=contrato_ocs_011_2023) ===
Saved: /content/drive/My Drive/KOA/onboarding/contratos_COM_UFO/KOA_UFO_contrato_ocs_011_2023.pl
Saved: /content/drive/My Drive/KOA/onboarding/contratos_COM_UFO/KOA_UFO_contrato_ocs_011_2023_trace.json

=== Processing KOA_Contrato_OCS_012_2022_-_VS_Data_Storages_de_b